# 4.2 – Producteur de flux – socket Python 1h – Présentiel

### Écrire un script Python simulant un flux de données taxi en émettant les lignes d'un CSV sur une socket TCP, sans charger le fichier en mémoire.



### 1. Lire le CSV ligne par ligne via un itérateur : with open("taxi.csv","r") as f: for line in f: ...


In [ ]:
with open("yellow_tripdata_2026-01.csv", "r") as f:
    for line in f:  # itérateur
        print(line)

### 2. Créer une socket TCP : socket.AF_INET, SOCK_STREAM, bind(("localhost", 9999)), listen, accept


In [ ]:
import socket

# 1. Créer le socket
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# 2. Évite l'erreur "Address already in use" si on relance le script
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

# 3. Attacher le socket au port 9999
server.bind(("localhost", 9999))

# 4. Mettre en écoute (max 1 client en attente)
server.listen(1)

print("En attente d'un client sur le port 9999...")

# 5. Accepter la connexion du client
client, address = server.accept()

print(f"Client connecté depuis {address}")

# teste
# Terminal 1 — lancer le serveur
#python3 producer.py

# Terminal 2 — se connecter
#nc localhost 9999

En attente d'un client sur le port 9999...
Client connecté depuis ('127.0.0.1', 40128)


### 3. Émettre chaque ligne encodée à 100 lignes/s : client.send((line+"\n").encode()); time.sleep(1/100)

In [5]:
import time

server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server.bind(("localhost", 9999))
server.listen(1)

print("En attente d'un client sur le port 9999...")
client, address = server.accept()
print(f"Client connecté depuis {address}")

with open("yellow_tripdata_2026-01.csv", "r") as f:
    for line in f:
        client.send((line + "\n").encode())  # envoie la ligne encodée
        time.sleep(1/100)                    # pause 10ms = 100 lignes/s

En attente d'un client sur le port 9999...
Client connecté depuis ('127.0.0.1', 43692)


BrokenPipeError: [Errno 32] Broken pipe

### 4. Tester la réception avec nc localhost 9999 dans un autre terminal

![emission_via_socket_100lignes_s.png](/home/youssef.hirchaou@Digital-Grenoble.local/campus/Big_data_architecture_distribuées/Streaming_Tabulaire/emission_via_socket_100lignes_s.png)

### 5. Vérifier que la RAM reste stable quel que soit le volume du fichier (htop)

In [26]:
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server.bind(("localhost", 9999))
server.listen(1)

print("En attente d'un client sur le port 9999...")
client, address = server.accept()
print(f"Client connecté depuis {address}")
try:
    with open("yellow_tripdata_2026-01.csv", "r") as f:
        for i, line in enumerate(f):
            client.send((line + "\n").encode())
            time.sleep(1/100)

            # Affiche la progression toutes les 1000 lignes
            if i % 1000 == 0:
                print(f"{i} lignes envoyées...")

except BrokenPipeError:
    print("Client déconnecté.")
finally:
    client.close()
    server.close()
    print("Socket fermé proprement.")

En attente d'un client sur le port 9999...
Client connecté depuis ('127.0.0.1', 34520)
0 lignes envoyées...
1000 lignes envoyées...
2000 lignes envoyées...
3000 lignes envoyées...
4000 lignes envoyées...
5000 lignes envoyées...
6000 lignes envoyées...
7000 lignes envoyées...
Client déconnecté.
Socket fermé proprement.


## 4.3 – Job Spark Structured Streaming + fenêtres glissantes

### Connexion à la source et parsing
### 6. Créer le DataFrame de streaming :
### spark.readStream.format("socket").option("host","localhost").option("port",9999).load()

In [50]:
from pyspark.sql import SparkSession

# Créer la session Spark
spark = SparkSession.builder \
    .appName("TaxiStreaming") \
    .getOrCreate()

# Réduire les logs pour y voir plus clair
spark.sparkContext.setLogLevel("WARN")

# Lire le flux depuis le socket
raw_df = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

# Vérification — doit afficher True
print("isStreaming :", raw_df.isStreaming)

isStreaming : True


26/04/23 10:51:12 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.


In [51]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_csv, col, window, count, avg, to_timestamp, expr


spark = SparkSession.builder \
    .appName("TaxiStreaming") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Schéma en string (obligatoire pour from_csv)
schema_str = """
    idx LONG,
    VendorID LONG,
    tpep_pickup_datetime STRING,
    tpep_dropoff_datetime STRING,
    passenger_count DOUBLE,
    trip_distance DOUBLE,
    RatecodeID DOUBLE,
    store_and_fwd_flag STRING,
    PULocationID LONG,
    DOLocationID LONG,
    payment_type LONG,
    fare_amount DOUBLE,
    extra DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    improvement_surcharge DOUBLE,
    total_amount DOUBLE,
    congestion_surcharge DOUBLE,
    Airport_fee DOUBLE,
    cbd_congestion_fee DOUBLE
"""

# Connexion au socket
raw_df = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()


26/04/23 10:51:13 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.


### 7. Parser les lignes CSV avec un StructType explicite et from_csv()

In [52]:
# Parser les lignes CSV
parsed_df = raw_df.select(
    from_csv(col("value"), schema_str).alias("data")
).select("data.*") \
 .withColumn("tpep_pickup_datetime", 
             expr("try_cast(tpep_pickup_datetime as timestamp)")) \
 .filter(col("tpep_pickup_datetime").isNotNull())  # ignore les lignes mal formées

###  8. Vérifier : df.isStreaming doit retourner True

In [53]:
print("isStreaming :", parsed_df.isStreaming)

isStreaming : True


## Agrégations sur fenêtres glissantes
### 9. Appliquer le watermarking : df.withWatermark("tpep_pickup_datetime","5 minutes")
### 10. Calculer sur une fenêtre de 2 minutes / slide 1 minute :

In [54]:
#  Watermark + fenêtres glissantes
agg_df = parsed_df \
    .withWatermark("tpep_pickup_datetime", "5 minutes") \
    .groupBy(
        window("tpep_pickup_datetime", "2 minutes", "1 minute"),
        "PULocationID"
    ) \
    .agg(
        count("*").alias("nb_trips"),
        avg("fare_amount").alias("avg_fare")
    )


### 11. Afficher en console pour valider : writeStream.format("console").outputMode("update").start()

In [55]:
# Affichage console pour valider
query = agg_df.writeStream \
    .format("console") \
    .outputMode("update") \
    .option("truncate", False) \
    .start()

query.awaitTermination()

26/04/23 10:51:21 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e5122a93-7805-4592-97af-d2ec9819ef5e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/23 10:51:21 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/23 10:51:21 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+------+------------+--------+--------+
|window|PULocationID|nb_trips|avg_fare|
+------+------------+--------+--------+
+------+------------+--------+--------+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+------------+--------+--------+
|window                                    |PULocationID|nb_trips|avg_fare|
+------------------------------------------+------------+--------+--------+
|{2026-01-01 00:50:00, 2026-01-01 00:52:00}|249         |2       |14.2    |
|{2026-01-01 00:28:00, 2026-01-01 00:30:00}|234         |1       |17.0    |
|{2026-01-01 00:16:00, 2026-01-01 00:18:00}|164         |1       |31.7    |
|{2026-01-01 00:21:00, 2026-01-01 00:23:00}|239         |2       |7.2     |
|{2026-01-01 00:46:00, 2026-01-01 00:48:00}|238         |2       |9.3     |
|{2026-01-01 00:58:00, 2026-01-01 01:00:00}|231         |1       |15.6    |
|{2026-01-01 00:23:00, 2026-01-01 00:25:00}|48          |1       |14.9    |
|{2026-01-01 00:31:00, 2026-01-01 00:33:00}|114         |1       |28.9    |
|{2026-01-01 00:56:00, 2026-01-01 00:58:00}|237         |1       |2

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+------------+--------+------------------+
|window                                    |PULocationID|nb_trips|avg_fare          |
+------------------------------------------+------------+--------+------------------+
|{2026-01-01 00:47:00, 2026-01-01 00:49:00}|237         |2       |13.850000000000001|
|{2026-01-01 00:50:00, 2026-01-01 00:52:00}|132         |1       |11.4              |
|{2026-01-01 00:46:00, 2026-01-01 00:48:00}|238         |3       |8.6               |
|{2026-01-01 00:46:00, 2026-01-01 00:48:00}|231         |1       |11.4              |
|{2026-01-01 00:56:00, 2026-01-01 00:58:00}|237         |2       |15.95             |
|{2026-01-01 00:30:00, 2026-01-01 00:32:00}|151         |1       |19.8              |
|{2026-01-01 00:03:00, 2026-01-01 00:05:00}|231         |1       |9.3               |
|{2026-01-01 00:13:00, 2026-01-01 00:15:00}

-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+------------+--------+------------------+
|window                                    |PULocationID|nb_trips|avg_fare          |
+------------------------------------------+------------+--------+------------------+
|{2026-01-01 00:58:00, 2026-01-01 01:00:00}|161         |1       |19.1              |
|{2026-01-01 00:56:00, 2026-01-01 00:58:00}|237         |3       |12.799999999999999|
|{2026-01-01 00:58:00, 2026-01-01 01:00:00}|262         |2       |9.3               |
|{2026-01-01 00:56:00, 2026-01-01 00:58:00}|239         |2       |19.8              |
|{2026-01-01 00:53:00, 2026-01-01 00:55:00}|137         |2       |13.149999999999999|
|{2026-01-01 00:56:00, 2026-01-01 00:58:00}|107         |1       |11.4              |
|{2026-01-01 00:56:00, 2026-01-01 00:58:00}|50          |2       |26.8              |
|{2026-01-01 00:55:00, 2026-01-01 00:57:00}

...

26/04/23 10:52:31 WARN TextSocketMicroBatchStream: Stream closed by localhost:9999


-------------------------------------------
Batch: 16
-------------------------------------------
+------------------------------------------+------------+--------+------------------+
|window                                    |PULocationID|nb_trips|avg_fare          |
+------------------------------------------+------------+--------+------------------+
|{2026-01-01 01:53:00, 2026-01-01 01:55:00}|164         |2       |19.45             |
|{2026-01-01 01:55:00, 2026-01-01 01:57:00}|236         |1       |12.1              |
|{2026-01-01 01:54:00, 2026-01-01 01:56:00}|234         |3       |33.1              |
|{2026-01-01 01:58:00, 2026-01-01 02:00:00}|48          |2       |19.1              |
|{2026-01-01 01:53:00, 2026-01-01 01:55:00}|107         |4       |9.3               |
|{2026-01-01 01:57:00, 2026-01-01 01:59:00}|141         |3       |11.4              |
|{2026-01-01 01:57:00, 2026-01-01 01:59:00}|114         |1       |7.2               |
|{2026-01-01 01:54:00, 2026-01-01 01:56:00

ERROR:root:KeyboardInterrupt while sending command.             (123 + 8) / 200]
Traceback (most recent call last):
  File "/home/youssef.hirchaou@Digital-Grenoble.local/campus/Big_data_architecture_distribuées/venv_bigdata/lib/python3.13/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/youssef.hirchaou@Digital-Grenoble.local/campus/Big_data_architecture_distribuées/venv_bigdata/lib/python3.13/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/home/youssef.hirchaou@Digital-Grenoble.local/anaconda3/lib/python3.13/socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 

-------------------------------------------
Batch: 17
-------------------------------------------
+------------------------------------------+------------+--------+------------------+
|window                                    |PULocationID|nb_trips|avg_fare          |
+------------------------------------------+------------+--------+------------------+
|{2026-01-01 01:56:00, 2026-01-01 01:58:00}|143         |2       |8.95              |
|{2026-01-01 01:57:00, 2026-01-01 01:59:00}|32          |1       |30.5              |
|{2026-01-01 01:56:00, 2026-01-01 01:58:00}|262         |1       |7.2               |
|{2026-01-01 01:54:00, 2026-01-01 01:56:00}|163         |1       |7.2               |
|{2026-01-01 01:53:00, 2026-01-01 01:55:00}|239         |1       |27.5              |
|{2026-01-01 01:57:00, 2026-01-01 01:59:00}|114         |2       |66.0              |
|{2026-01-01 01:56:00, 2026-01-01 01:58:00}|32          |1       |30.5              |
|{2026-01-01 01:55:00, 2026-01-01 01:57:00